# SPY Trend-Following Backtest (12 Years)

**What this does:** Detects when SPY flips between uptrend and downtrend using pivot highs/lows on the **weekly** chart (way less noise than daily), then backtests a simple strategy:
- **Uptrend signal** (higher highs + higher lows) = Buy SPY
- **Downtrend signal** (lower highs + lower lows) = Sell SPY, hold cash

Compares the strategy against just holding SPY for 12 years.

**How to use:** Just tap the Play button on the cell below. That's it!

In [ ]:
# === STEP 1: Install & import (takes ~10 seconds) ===
!pip install yfinance -q

import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

# === SETTINGS (change these if you want) ===
TICKER = "SPY"              # What to trade
YEARS = 12                  # How many years to backtest
STARTING_CASH = 10000       # Starting portfolio value
TIMEFRAME = "weekly"        # "weekly" (less noise) or "daily"
K = 3                       # Pivot lookback window (3 for weekly, 5 for daily)

# === STEP 2: Download data ===
# Grab extra data so we have enough history for pivot detection
start_date = (datetime.now() - timedelta(days=YEARS * 365 + 365)).strftime("%Y-%m-%d")
print(f"Downloading {TICKER} data from {start_date}...")
df_daily = yf.download(TICKER, start=start_date, auto_adjust=False, progress=False)
if isinstance(df_daily.columns, pd.MultiIndex):
    df_daily.columns = df_daily.columns.get_level_values(0)
df_daily = df_daily.dropna().sort_index()
print(f"Got {len(df_daily)} daily bars ({df_daily.index[0].date()} to {df_daily.index[-1].date()})")

# Resample to weekly if requested
if TIMEFRAME == "weekly":
    df = df_daily.resample("W-FRI").agg({
        "Open": "first", "High": "max", "Low": "min", "Close": "last"
    }).dropna()
    print(f"Resampled to {len(df)} weekly bars")
    PERIODS_PER_YEAR = 52
else:
    df = df_daily.copy()
    PERIODS_PER_YEAR = 252

# === STEP 3: Find pivot highs & lows ===
highs, lows = df["High"].values, df["Low"].values
ph, pl = np.zeros(len(df), bool), np.zeros(len(df), bool)
for i in range(K, len(df) - K):
    wh = highs[i-K:i+K+1]
    wl = lows[i-K:i+K+1]
    if highs[i] == wh.max() and wh.argmax() == K: ph[i] = True
    if lows[i] == wl.min() and wl.argmin() == K: pl[i] = True
pivots = df.loc[ph | pl, ["High", "Low"]].copy()
pivots["pivot_high"], pivots["pivot_low"] = ph[ph | pl], pl[ph | pl]
print(f"Found {pivots['pivot_high'].sum()} pivot highs, {pivots['pivot_low'].sum()} pivot lows (k={K}, {TIMEFRAME})")

# === STEP 4: Detect trend inflections (THE KEY PART) ===
h2, l2, regime, flips = [], [], None, []
for dt, r in pivots.iterrows():
    if r["pivot_high"]: h2.append((dt, r["High"])); h2 = h2[-2:]
    if r["pivot_low"]:  l2.append((dt, r["Low"]));  l2 = l2[-2:]
    if len(h2) == 2 and len(l2) == 2:
        up = h2[1][1] > h2[0][1] and l2[1][1] > l2[0][1]   # Higher high + higher low
        dn = h2[1][1] < h2[0][1] and l2[1][1] < l2[0][1]   # Lower high + lower low
        nr = "uptrend" if up else "downtrend" if dn else regime
        if nr != regime and nr:
            flips.append({"date": dt, "regime": nr})
            regime = nr
flips = pd.DataFrame(flips)

# ========================================
# HERE'S YOUR LIST OF INFLECTION DATES
# ========================================
print(f"\n{'='*55}")
print(f"  {len(flips)} TREND INFLECTION POINTS ({TIMEFRAME} chart, k={K})")
print(f"{'='*55}")
for _, f in flips.iterrows():
    arrow = "▲ UPTREND" if f["regime"] == "uptrend" else "▼ DOWNTREND"
    print(f"  {pd.Timestamp(f['date']).strftime('%Y-%m-%d')}  {arrow}")

# === STEP 5: Run the backtest (uses daily bars for realistic fills) ===
bt_start = datetime.now() - timedelta(days=YEARS * 365)
bt_df = df_daily.loc[df_daily.index >= pd.Timestamp(bt_start)].copy()
bt_flips = flips[pd.to_datetime(flips["date"]) >= pd.Timestamp(bt_start)]

# Build signal lookup
signals = {pd.Timestamp(r["date"]): r["regime"] for _, r in bt_flips.iterrows()}

# Simulate
cash = STARTING_CASH
shares = 0.0
pending = None
strat_vals = []
trade_log = []

for dt in bt_df.index:
    price = bt_df.loc[dt, "Close"]
    if isinstance(price, pd.Series): price = price.iloc[0]

    # Execute yesterday's signal at today's close
    if pending == "uptrend" and shares == 0:
        shares = cash / price
        trade_log.append({"date": dt, "action": "BUY", "price": round(price, 2), "value": round(cash, 2)})
        cash = 0.0
    elif pending == "downtrend" and shares > 0:
        val = shares * price
        trade_log.append({"date": dt, "action": "SELL", "price": round(price, 2), "value": round(val, 2)})
        cash = val
        shares = 0.0
    pending = None

    if dt in signals:
        pending = signals[dt]

    strat_vals.append(cash + shares * price)

# Buy-and-hold
first_price = bt_df.iloc[0]["Close"]
if isinstance(first_price, pd.Series): first_price = first_price.iloc[0]
bh_shares = STARTING_CASH / first_price
bh_vals = [(bh_shares * (bt_df.loc[dt, "Close"].iloc[0] if isinstance(bt_df.loc[dt, "Close"], pd.Series) else bt_df.loc[dt, "Close"])) for dt in bt_df.index]

equity = pd.DataFrame({"Strategy": strat_vals, "Buy_Hold": bh_vals}, index=bt_df.index)

# === STEP 6: Print results ===
def stats(vals, name):
    s = pd.Series(vals, index=bt_df.index)
    years = (s.index[-1] - s.index[0]).days / 365.25
    total_ret = (s.iloc[-1] / s.iloc[0] - 1) * 100
    cagr = ((s.iloc[-1] / s.iloc[0]) ** (1/years) - 1) * 100
    dd = ((s - s.cummax()) / s.cummax()).min() * 100
    daily = s.pct_change().dropna()
    sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0
    print(f"  {name}")
    print(f"    Final value:    ${s.iloc[-1]:,.0f}")
    print(f"    Total return:   {total_ret:+.1f}%")
    print(f"    Annual return:  {cagr:.1f}%")
    print(f"    Max drawdown:   {dd:.1f}%")
    print(f"    Sharpe ratio:   {sharpe:.2f}")
    print()

print(f"\n{'='*55}")
print(f"  BACKTEST: {bt_df.index[0].date()} to {bt_df.index[-1].date()}")
print(f"  Signals from {TIMEFRAME} chart (k={K}) | Fills on daily bars")
print(f"  Starting with ${STARTING_CASH:,}")
print(f"{'='*55}\n")
stats(strat_vals, "TREND STRATEGY (buy uptrend / sell downtrend)")
stats(bh_vals, "BUY & HOLD SPY")

# Trade log
trades = pd.DataFrame(trade_log)
if not trades.empty:
    trades["date"] = pd.to_datetime(trades["date"]).dt.strftime("%Y-%m-%d")
    buys = len(trades[trades["action"] == "BUY"])
    sells = len(trades[trades["action"] == "SELL"])
    print(f"  TRADES: {buys} buys, {sells} sells ({buys + sells} total)")
    print(f"{'='*55}\n")
    print(trades.to_string(index=False))

# === STEP 7: Draw chart ===
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(equity.index, equity["Strategy"], label="Trend Strategy", linewidth=2, color="#2196F3")
    ax.plot(equity.index, equity["Buy_Hold"], label="Buy & Hold SPY", linewidth=1.5, color="gray", alpha=0.7)

    # Mark buy/sell points
    if not trades.empty:
        for _, t in pd.DataFrame(trade_log).iterrows():
            color = "green" if t["action"] == "BUY" else "red"
            marker = "^" if t["action"] == "BUY" else "v"
            ax.scatter(t["date"], t["value"], color=color, marker=marker, s=60, zorder=5)

    ax.set_title(f"{TICKER} Trend Strategy vs Buy & Hold ({YEARS}yr, {TIMEFRAME} signals k={K})", fontsize=14)
    ax.set_ylabel("Portfolio Value ($)")
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
except Exception as e:
    print(f"(Chart skipped: {e})")

print("\nDone! Scroll up to see your results.")